# Forecasting Analysis

This notebook reproduces the forecasting workflow for the retail dataset and mirrors the logic in `src/forecasting.py`.

It evaluates two short-term forecasting approaches:
- Seasonal Naive
- Additive Holt-Winters

The lower-RMSE model is used for the forward 30-day forecast.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

sys.path.append(str(root / "src"))

from forecasting import (
    DATA_PATH,
    FORECAST_DAYS,
    TEST_DAYS,
    evaluate_target,
    load_daily_targets,
    render_table,
    summarize_forecast,
)

## Load And Prepare Daily Targets

The cleaned transactional dataset is aggregated into two daily time series:
- `daily_revenue`
- `daily_order_volume`

Missing dates are kept and treated as zero-activity days so the model can learn the weekly operating pattern.

In [ ]:
dates, targets = load_daily_targets(DATA_PATH)

print(f"Observations: {len(dates)} daily records")
print(f"Date range: {dates[0].isoformat()} to {dates[-1].isoformat()}")
print("Targets:", ", ".join(targets.keys()))
print(f"Holdout window: last {TEST_DAYS} days")
print(f"Forecast horizon: next {FORECAST_DAYS} days")

## Evaluate Competing Forecast Models

Each target is split chronologically:
- training set: all observations except the final 30 days
- test set: the final 30 days

The notebook scores both models with MAE, RMSE, and WAPE.

In [ ]:
all_metrics = []
all_forecasts = []
all_summaries = []

for target_name, series in targets.items():
    target_metrics, target_forecasts = evaluate_target(target_name, series, dates)
    all_metrics.extend(target_metrics)
    all_forecasts.extend(target_forecasts)
    all_summaries.append(
        summarize_forecast(
            target_name,
            target_forecasts[0]["model"],
            [row["forecast"] for row in target_forecasts],
        )
    )

print("Model evaluation:")
print(render_table(all_metrics, ["target", "model", "mae", "rmse", "wape"]))

## Forecast Summary

The forward forecast uses the lower-RMSE model for each target. In the current dataset, Seasonal Naive is selected because it outperforms Holt-Winters on both revenue and order volume.

In [ ]:
print("30-day forecast summary:")
print(
    render_table(
        all_summaries,
        [
            "target",
            "selected_model",
            "30_day_total_forecast",
            "average_daily_forecast",
            "peak_day_forecast",
        ],
    )
)

preview_rows = []
seen_by_target = {}
for row in all_forecasts:
    seen_by_target.setdefault(row["target"], 0)
    if seen_by_target[row["target"]] < 7:
        preview_rows.append(row)
        seen_by_target[row["target"]] += 1

print()
print("Next 7 forecasted days:")
print(render_table(preview_rows, ["date", "target", "model", "forecast"]))

## Operational Impact

Short-term forecasts from this notebook can be used to:
- plan staffing for order-processing peaks
- prepare replenishment for higher-demand days
- estimate near-term revenue for tactical cash-flow decisions

The main limitation is that the model only sees transaction history. Promotions, holidays, and supply disruptions are not included, so the forecasts should be used as tactical guidance rather than a long-range plan.